In [ ]:
#####################################################################################
##### Generate random sample points for SCUBA surveys & format for easy export ######
#####################################################################################

#script is saved as a notebook in ArcGIS Pro 

#set up environment and connect to source data 

import arcpy
import re
import os

# Local workspace (change this to your geodatabase or scratch workspace)
workspace = r"C:\Users\bparadis\Documents\ArcGIS\Projects\OSMonitoring\OSMonitoring.gdb"
arcpy.env.workspace = workspace

# connect to feature service layer (needs to end in /0)
url = "https://services2.arcgis.com/kCu40SDxsCGcuUWO/arcgis/rest/services/OS_material_10_2024/FeatureServer/0"
#layer name
original_layer = "OS_material_layer"
#create a feature layer in map
arcpy.MakeFeatureLayer_management(url, original_layer)

#nested dictionary for specifying sample size at each material/site
sample_size = {
    'Croatan Sound' : {'Marl':4, 'Reef Balls':4},
    'Deep Bay' : {'Marl':4, 'Reef Balls':4},
    'West Bay' : {'Marl':4, 'Reef Balls':4},
    'Crab Hole' : {'Marl':8},
    'Middle Bay' : {'Marl':4},
    'Neuse River' : {'Marl':4},
    'West Bluff' : {'Marl':4, 'Reef Balls':4},
    'Gibbs Shoal' : {'Marl':5, 'Reef Balls':4},
    'Long Shoal' : {'Reef Balls':4},
    'Raccoon Island' : {'Crushed Concrete':4, 'Consolidated Concrete':4, 'Reef Balls': 4},
    'Pea Island' : {'Crushed Concrete':4, 'Consolidated Concrete':4, 'Reef Balls': 4},
    'Little Creek' : {'Marl':4, 'Reef Balls':4, 'Basalt':4, 'Crushed Concrete':4, 'Consolidated Concrete':4, 'Reef Balls': 4, 'Granite':4},
    'Swan Island' : {'Marl': 8, 'Granite': 7},
    'Cedar Island' : {'Marl':8, 'Crushed Concrete':4},
    'Maw Point' : {'Marl':12},
    'Brant Island' : {'Granite': 10, 'Crushed Concrete':5}
}

# Coordinate formatting function
def decimal_to_ddm(decimal_coord, is_longitude=False):
    """
    Converts a decimal degree coordinate to Degree Decimal Minutes (DDM) format.
    Example: 35.67891° → 35° 40.735' N
    """
    degrees = int(decimal_coord)  # Extract the whole degrees
    minutes = abs(decimal_coord - degrees) * 60  # Convert remainder to minutes
    
    #cardinal direction
    if is_longitude:
        cardinal = 'W' if decimal_coord < 0 else 'E'
    else:
        cardinal = 'S' if decimal_coord < 0 else 'N'
        
    return f"{abs(degrees)}° {round(minutes, 3)}' {cardinal}"  # Format as "DD MM.MMM' N/W"



In [ ]:
#main block for processing each site/material and generating random points

#set target folder for CSV export (change this to your desired output location)
csv_folder = r"C:\Users\bparadis\Documents\ArcGIS\Projects\OSMonitoring\2026 monitoring points"

target_site = input("Enter the site name to process (leave blank for all sites): ").strip()
if target_site == "":
    target_site = None

#Get the spatial reference from the source layer 
spatial_ref = arcpy.Describe(original_layer).spatialReference

#Create or update the output feature class 
output_fc_name = "Monitoring_Points_2026"
output_fc = os.path.join(workspace, output_fc_name)

if arcpy.Exists(output_fc):
    if target_site:
        # Delete only rows belonging to the target site, keep everything else
        with arcpy.da.UpdateCursor(output_fc, ["Site_Name"]) as cursor:
            for row in cursor:
                if row[0] == target_site:
                    cursor.deleteRow()
        print(f"Cleared existing rows for '{target_site}' from {output_fc_name}")
    else:
        # No target site specified — full run, so wipe and recreate
        arcpy.management.Delete(output_fc)
        print(f"Deleted existing {output_fc_name} for full rebuild")

if not arcpy.Exists(output_fc):
    spatial_ref = arcpy.Describe(original_layer).spatialReference
    arcpy.management.CreateFeatureclass(
        out_path=workspace,
        out_name=output_fc_name,
        geometry_type="POINT",
        spatial_reference=spatial_ref
    )
    for field, f_type, length in [
        ("Site_Name", "TEXT",  50),
        ("Sample",    "SHORT",  2),
        ("Material",  "TEXT",  50),
        ("Longitude", "TEXT",  30),
        ("Latitude",  "TEXT",  30),
    ]:
        arcpy.management.AddField(output_fc, field, f_type, field_length=length)
    print(f"Created output feature class: {output_fc}")

#Process each site/material and insert points directly into output Feature Class

for site_name, materials in sample_size.items():
    quad_num = 1
    
    if target_site and site_name != target_site:
        continue

    for material, num_points in materials.items():

        material_query = f"OS_Site = '{site_name}' AND Material = '{material}'"
        material_layer = f"tmp_material_layer"

        # Clean up temp layers/FCs before each iteration
        if arcpy.Exists(material_layer):
            arcpy.management.Delete(material_layer)

        dissolved_fc = os.path.join(workspace, "tmp_dissolved")
        if arcpy.Exists(dissolved_fc):
            arcpy.management.Delete(dissolved_fc)

        tmp_points = os.path.join(workspace, "tmp_random_points")
        if arcpy.Exists(tmp_points):
            arcpy.management.Delete(tmp_points)

        print(f"Generating {num_points} random points for {material} at {site_name}")

        try:
            arcpy.MakeFeatureLayer_management(original_layer, material_layer, material_query)
            arcpy.Dissolve_management(material_layer, dissolved_fc, multi_part="MULTI_PART")

            # Generate random points into a temporary Feature Class
            arcpy.management.CreateRandomPoints(
                out_path=workspace,
                out_name="tmp_random_points",
                constraining_feature_class=dissolved_fc,
                number_of_points_or_field=num_points,
                minimum_allowed_distance= "2 meters"
            )

            # Read temp points and insert into the single output feature class with attributes
            insert_fields = ["SHAPE@XY", "Site_Name", "Sample", "Material",
                             "Longitude", "Latitude"]

            with arcpy.da.SearchCursor(tmp_points, ["SHAPE@XY"]) as s_cursor, \
                 arcpy.da.InsertCursor(output_fc, insert_fields) as i_cursor:
                for row in s_cursor:
                    
                    x, y = row[0]
                    i_cursor.insertRow([
                        (x, y),
                        site_name,
                        quad_num,
                        material,
                        decimal_to_ddm(x, is_longitude=True),
                        decimal_to_ddm(y, is_longitude=False),
                    ])
                    quad_num +=1

            print(f"  Inserted {num_points} points for {material} at {site_name}")

        except Exception as e:
            print(f"  ERROR for {material} at {site_name}: {e}")

#Clean up remaining temporary layers/feature classes
for tmp in ["tmp_material_layer", "tmp_dissolved", "tmp_random_points"]:
    if arcpy.Exists(tmp):
        arcpy.management.Delete(tmp)

#Export single combined CSV with all points and attributes for easy reference in the field
csv_output_path = os.path.join(csv_folder, f"{output_fc_name}.csv")
arcpy.conversion.ExportTable(output_fc, csv_output_path)
print(f"\nCSV exported: {csv_output_path}")
print(f"Done — {quad_num - 1} total points written to {output_fc}")